In [1]:
# Parameters
Month = 3
Year = 2569


In [2]:
# pip install python-calamine
import time  # <--- 1. import time
import pandas as pd
import os
import glob
import re 
start_time = time.time() 
print("Start processing...")

Start processing...


In [3]:
import time
import pandas as pd
import os
from pathlib import Path

def run_auto_sync():
    start_time = time.time()
    current_dir = Path.cwd() 
    base_database = current_dir.parent / "01-database" / "Shop_BKK"
    
    print(f"🚀 [Auto-Scan] Target Path: {base_database}")

    if not base_database.exists():
        print(f"❌ [Error] Path not found: {base_database}")
        return None # Return None ถ้าไม่เจอ Path

    all_data = []
    extensions = ['*.xlsx', '*.xlsb', '*.xlsm']
    files = []
    for ext in extensions:
        files.extend(list(base_database.rglob(ext)))

    if not files:
        print("⚠️ No Excel files found in Shop_BKK.")
        return None

    print(f"📂 Found {len(files)} files. Reading with Calamine...")

    for file in files:
        try:
            df = pd.read_excel(file,  skiprows=2, engine='calamine')
            df['folder_name'] = file.parent.name
            df['file_source'] = file.name
            all_data.append(df)
            print(f"   ✅ Loaded: {file.name}")
        except Exception as e:
            print(f"   ⚠️ Skip {file.name}: {e}")

    if all_data:
        # รวม Dataframe
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # บันทึกไฟล์
        output_path = current_dir / "Auto_Combined_Shop_BKK.xlsx"
        # combined_df.to_excel(output_path, index=False)
        
        print("-" * 30)
        print(f"✨ SUCCESS! Total Rows: {len(combined_df)}")
        print(f"⏱️ Time: {time.time() - start_time:.2f} sec")
        
        return combined_df # <--- เพิ่มบรรทัดนี้เพื่อส่งค่าออกไป
    else:
        print("❌ No data to combine.")
        return None

# --- เรียกใช้งานและนำตัวแปร df มารับค่า ---
dshop = run_auto_sync()

# # แสดงผลข้อมูล
# if dshop is not None:
#     display(dshop.head()) # ใช้ display() ใน Notebook จะสวยกว่า print()
dshop.head(2)    

🚀 [Auto-Scan] Target Path: c:\Users\User\Documents\GitHub\AutomationCPI\01-database\Shop_BKK
📂 Found 1 files. Reading with Calamine...
   ✅ Loaded: shop_data.xlsx
------------------------------
✨ SUCCESS! Total Rows: 5148
⏱️ Time: 0.11 sec


,ลำดับที่,รหัสร้าน,ชื่อร้าน,ประเภท,ชื่อผู้ติดต่อ,เบอร์โทรศัพท์,Fax,ที่อยู่,แขวง/ตำบล,เขต/อำเภอ,จังหวัด,รหัสไปรษณีย์,กลุ่ม,พิกัด GPS,กิจกรรม,บันทึกช่วยจำ,folder_name,file_source
0,1,100100001,ร้านขายเสื้อผ้าสำเร็จรูป,ร้านค้าทั่วไป,libing,"02-971-0596, 083-289-7915",NaN,- ในตลาดพาหุรัด,บางขุนพรหม,พระนคร,กรุงเทพมหานคร,NaN,- กลุ่ม วงเวียนใหญ่,NaN,NaN,NaN,Shop_BKK,shop_data.xlsx
1,2,100100002,ร้านขายกางเกง (คุณสุ),ร้านค้าทั่วไป,123,123,NaN,- ในตลาดพาหุรัด,พระบรมมหาราชวัง,พระนคร,กรุงเทพมหานคร,NaN,- กลุ่ม วงเวียนใหญ่,NaN,NaN,NaN,Shop_BKK,shop_data.xlsx


In [4]:
# ShopName = pd.read_excel('shop_data_20260105_105356.xlsx', engine='calamine')
files_shop = glob.glob("shop_data*.xlsx")
ds_ = dshop
# Dealer Group
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
# df = df.merge(ds, on=["DILLER_CODE"], how="left")
ds

,DILLER_CODE,กลุ่ม
0,0100100001,กลุ่มวงเวียนใหญ่
1,0100100002,กลุ่มวงเวียนใหญ่
2,0100100003,กลุ่มพรานนก
3,0100100004,กลุ่มวงเวียนใหญ่
4,0100100005,กลุ่มวงเวียนใหญ่
...,...,...
5143,0100500292,กลุ่มสะพานใหม่
5144,0100500293,กลุ่มสะพานใหม่
5145,0102400213,กลุ่มบางประกอก
5146,0100633441,กลุ่มบางกะปิ


In [5]:
# os.getcwd()


In [6]:
import os
import sys
from pathlib import Path

# 1. กำหนดค่า Input ที่ต้องการ
target_year = 2569
target_month = 2

current_dir = Path.cwd() 
base_database = current_dir.parent / "01-database" / "Data_G_Month"
folder_path = base_database

# 2. ดึงรายชื่อไฟล์และกรองเบื้องต้น
if not folder_path.exists():
    print(f"❌ ไม่พบโฟลเดอร์: {folder_path}")
    sys.exit()

all_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]
unwanted = {'pivot_compare_PriceRelComm.xlsx', 'Test.xlsx', 'Test3.xlsx', 'SeriesG102568.xslx', 'TimeSeries_FULL.xlsx'}

def get_ym_tuple(filename):
    try:
        parts = filename.replace('.xlsx', '').split('_')[-1].split('-')
        return int(parts[0]), int(parts[1])
    except:
        return (0, 0)

excel_files = [
    f for f in all_files 
    if f not in unwanted 
    and not f.startswith(('~', 'Series')) 
    and f.lower().startswith('cpig')
]

# 3. กรองเฉพาะที่เก่ากว่า target และเรียงลำดับ
excel_files = [f for f in excel_files if get_ym_tuple(f) < (target_year, target_month)]
excel_files = sorted(excel_files, key=get_ym_tuple)

# 4. --- ตรวจสอบการข้ามเดือน (Strict & Gap Detection) ---
if not excel_files:
    print("❌ ไม่พบไฟล์ที่ตรงตามเงื่อนไข")
    sys.exit()

# --- ตรวจสอบช่องว่างจากไฟล์ล่าสุดไปจนถึง Target ---
last_y, last_m = get_ym_tuple(excel_files[-1])
target_total_months = (target_year * 12) + target_month
last_total_months = (last_y * 12) + last_m

# คำนวณส่วนต่างเดือน
skip_count = target_total_months - last_total_months

# ถ้าส่วนต่างมากกว่า 1 เดือน (เช่น มีแค่เดือน 1 แต่จะเอาเดือน 5 ส่วนต่างคือ 4)
if skip_count > 1:
    print(f"⚠️  พบการข้ามเดือน: ไฟล์ล่าสุดที่มีคือ {last_m}/{last_y}")
    print(f"ข้ามมา {skip_count} เดือน")
    print("🛑 หยุดการทำงาน...")
    sys.exit()

# --- ตรวจสอบการข้ามเดือนภายในช่วงข้อมูล (กรณีมีรูโหว่ตรงกลาง) ---
available_dates = {get_ym_tuple(f) for f in excel_files}
start_y, start_m = get_ym_tuple(excel_files[0])
curr_y, curr_m = start_y, start_m

while (curr_y, curr_m) <= (last_y, last_m):
    if (curr_y, curr_m) not in available_dates:
        print(f"⚠️  พบการข้ามเดือนภายในชุดข้อมูล: ตรวจไม่พบไฟล์เดือน {curr_m}/{curr_y}")
        print("🛑 หยุดการทำงาน...")
        sys.exit()
    
    curr_m += 1
    if curr_m > 12:
        curr_m = 1
        curr_y += 1

print(f"✅ ตรวจสอบความต่อเนื่องครบถ้วน: พบทั้งหมด {len(excel_files)} ไฟล์")
print(f"📂 ไฟล์ล่าสุดคือ: {excel_files[-1]}")

✅ ตรวจสอบความต่อเนื่องครบถ้วน: พบทั้งหมด 37 ไฟล์
📂 ไฟล์ล่าสุดคือ: cpig_month_2569-1.xlsx


In [7]:
# folder_path = os.getcwd()
import os
import sys
from pathlib import Path

# 1. กำหนดค่า Input ที่ต้องการ
target_year = Year
target_month = Month

target_year = int(target_year)
target_month = int(target_month)

current_dir = Path.cwd() 
base_database = current_dir.parent / "01-database" / "Data_G_Month"
folder_path = base_database

# 2. ดึงรายชื่อไฟล์และกรองเบื้องต้น
if not folder_path.exists():
    print(f"❌ ไม่พบโฟลเดอร์: {folder_path}")
    sys.exit()

all_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]
unwanted = {'pivot_compare_PriceRelComm.xlsx', 'Test.xlsx', 'Test3.xlsx', 'SeriesG102568.xslx', 'TimeSeries_FULL.xlsx'}

def get_ym_tuple(filename):
    try:
        parts = filename.replace('.xlsx', '').split('_')[-1].split('-')
        return int(parts[0]), int(parts[1])
    except:
        return (0, 0)

excel_files = [
    f for f in all_files 
    if f not in unwanted 
    and not f.startswith(('~', 'Series')) 
    and f.lower().startswith('cpig')
]

# 3. กรองเฉพาะที่เก่ากว่า target และเรียงลำดับ
excel_files = [f for f in excel_files if get_ym_tuple(f) < (target_year, target_month)]
excel_files = sorted(excel_files, key=get_ym_tuple)

# 4. --- ตรวจสอบการข้ามเดือน (Strict & Gap Detection) ---
if not excel_files:
    print("❌ ไม่พบไฟล์ที่ตรงตามเงื่อนไข")
    sys.exit()

# --- ตรวจสอบช่องว่างจากไฟล์ล่าสุดไปจนถึง Target ---
last_y, last_m = get_ym_tuple(excel_files[-1])
target_total_months = (target_year * 12) + target_month
last_total_months = (last_y * 12) + last_m

# คำนวณส่วนต่างเดือน
skip_count = target_total_months - last_total_months

# ถ้าส่วนต่างมากกว่า 1 เดือน (เช่น มีแค่เดือน 1 แต่จะเอาเดือน 5 ส่วนต่างคือ 4)
if skip_count > 1:
    print(f"⚠️  พบการข้ามเดือน: ไฟล์ล่าสุดที่มีคือ {last_m}/{last_y}")
    print(f"ข้ามมา {skip_count} เดือน")
    print("🛑 หยุดการทำงาน...")
    sys.exit()

# --- ตรวจสอบการข้ามเดือนภายในช่วงข้อมูล (กรณีมีรูโหว่ตรงกลาง) ---
available_dates = {get_ym_tuple(f) for f in excel_files}
start_y, start_m = get_ym_tuple(excel_files[0])
curr_y, curr_m = start_y, start_m

while (curr_y, curr_m) <= (last_y, last_m):
    if (curr_y, curr_m) not in available_dates:
        print(f"⚠️  พบการข้ามเดือนภายในชุดข้อมูล: ตรวจไม่พบไฟล์เดือน {curr_m}/{curr_y}")
        print("🛑 หยุดการทำงาน...")
        sys.exit()
    
    curr_m += 1
    if curr_m > 12:
        curr_m = 1
        curr_y += 1

print(f"✅ ตรวจสอบความต่อเนื่องครบถ้วน: พบทั้งหมด {len(excel_files)} ไฟล์")
print(f"📂 ไฟล์ล่าสุดคือ: {excel_files[-1]}")
# --- ส่วนอ่านข้อมูล (ปรับปรุงใหม่ให้เร็วขึ้น) ---
dataframes = []
selected_columns = [
    'TYPE', 'G_FLAG', 'L_FLAG', 'R_FLAG', 'SHOP_NAME', 'PROVINCE_NAME',
    'DESCRIPTION', 'COMMODITY_CODE', 'DILLER_CODE', 'MONTH', 'YEAR', 'AVG', 'REL','COMPARE_PRICE'
]

# กำหนด Type ล่วงหน้าเพื่อลดเวลา convert และบังคับให้อ่านเป็น string
col_types = {
    'MONTH': str, 
    'YEAR': str, 
    'COMMODITY_CODE': str, 
    'DILLER_CODE': str
}

print(f"Processing {len(excel_files)} files...")

for file in excel_files:
    file_path = os.path.join(folder_path, file)
    # KEY OPTIMIZATION: ใช้ usecols เพื่อโหลดเฉพาะคอลัมน์ที่ใช้ และ dtype เพื่อลดการแปลงข้อมูล
    # df = pd.read_excel(file_path, usecols=selected_columns, dtype=col_types)
    df = pd.read_excel(file_path, usecols=selected_columns, dtype=col_types, engine='calamine')
    dataframes.append(df)
    print(f"Loaded: {file}")

# --- รวมและจัดการข้อมูล ---
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)

    # จัดการ Format String ทีเดียว (Vectorized operation เร็วกว่าทำใน loop)
    combined_df['MONTH'] = combined_df['MONTH'].str.zfill(2)
    combined_df['COMMODITY_CODE'] = combined_df['COMMODITY_CODE'].str.zfill(16)
    combined_df['DILLER_CODE'] = combined_df['DILLER_CODE'].str.zfill(10)

    # เรียงลำดับ
    combined_df = combined_df.sort_values(
        by=['TYPE', 'COMMODITY_CODE', 'DILLER_CODE', 'YEAR', 'MONTH']
    ).reset_index(drop=True)

    # คัดลอกไปยัง clipboard
    combined_df.to_clipboard()

    # แสดงผล
    display(combined_df.head(2))
else:
    print("No matching files found.")

✅ ตรวจสอบความต่อเนื่องครบถ้วน: พบทั้งหมด 38 ไฟล์
📂 ไฟล์ล่าสุดคือ: cpig_month_2569-2.xlsx
Processing 38 files...


Loaded: cpig_month_2566-1.xlsx
Loaded: cpig_month_2566-2.xlsx
Loaded: cpig_month_2566-3.xlsx
Loaded: cpig_month_2566-4.xlsx
Loaded: cpig_month_2566-5.xlsx
Loaded: cpig_month_2566-6.xlsx
Loaded: cpig_month_2566-7.xlsx
Loaded: cpig_month_2566-8.xlsx
Loaded: cpig_month_2566-9.xlsx
Loaded: cpig_month_2566-10.xlsx
Loaded: cpig_month_2566-11.xlsx
Loaded: cpig_month_2566-12.xlsx
Loaded: cpig_month_2567-1.xlsx
Loaded: cpig_month_2567-2.xlsx
Loaded: cpig_month_2567-3.xlsx
Loaded: cpig_month_2567-4.xlsx
Loaded: cpig_month_2567-5.xlsx
Loaded: cpig_month_2567-6.xlsx
Loaded: cpig_month_2567-7.xlsx
Loaded: cpig_month_2567-8.xlsx
Loaded: cpig_month_2567-9.xlsx
Loaded: cpig_month_2567-10.xlsx
Loaded: cpig_month_2567-11.xlsx
Loaded: cpig_month_2567-12.xlsx
Loaded: cpig_month_2568-1.xlsx
Loaded: cpig_month_2568-2.xlsx
Loaded: cpig_month_2568-3.xlsx
Loaded: cpig_month_2568-4.xlsx
Loaded: cpig_month_2568-5.xlsx
Loaded: cpig_month_2568-6.xlsx
Loaded: cpig_month_2568-7.xlsx
Loaded: cpig_month_2568-8.xlsx
Lo

,TYPE,COMMODITY_CODE,MONTH,YEAR,DILLER_CODE,AVG,G_FLAG,L_FLAG,R_FLAG,COMPARE_PRICE,REL,SHOP_NAME,PROVINCE_NAME,DESCRIPTION
0,10,1112101111010201,01,2566,0100100010,25.0,0,1,0,25.0,100.0,ห้างตั้งฮั่วเส็งบางลำพู(เปลี่ยนเป็นตั้งฮั๋วเส็...,กรุงเทพมหานคร,แป้งข้าวเจ้า บรรจุถุง\ น้ำหนัก 500 กรัม\ ตราเห...
1,10,1112101111010201,02,2566,0100100010,25.0,0,1,0,25.0,100.0,ห้างตั้งฮั่วเส็งบางลำพู(เปลี่ยนเป็นตั้งฮั๋วเส็...,กรุงเทพมหานคร,แป้งข้าวเจ้า บรรจุถุง\ น้ำหนัก 500 กรัม\ ตราเห...


In [8]:
# ==========================================
# 1. เตรียมข้อมูลและจัด Format เบื้องต้น
# ==========================================
df = combined_df.copy()

df["COMMODITY_CODE"] = df["COMMODITY_CODE"].astype(str).str.zfill(16)
df["DILLER_CODE"]    = df["DILLER_CODE"].astype(str).str.zfill(10)
df["CODE_7_DIGITS"]  = df["COMMODITY_CODE"].astype(str).str[:7]
df["CODE9"] = df["COMMODITY_CODE"].str[7:16]
# ==========================================
# 2. จัดการประเภทและขนาดร้านค้า
# ==========================================

def classify_shop_type(x):
    if not isinstance(x, str): return "ร้านอื่นๆ"
    s = x 
    if any(k in s for k in ["โลตัส", "tesco", "Tesco", "โลตัล", "Lotus"]): return "Tesco Lotus"
    if any(k in s for k in ["บิ๊กซี", "บิกซี", "Big C", "Big-C"]): return "Big C"
    if any(k in s for k in ["อีเลฟเว่น", "ELEVEN", "7 - 11", "7-11", "7-ELEVEN", "เซเว่น", "7-Eleven", "seven", "SEVEN", "Eleven", "เซ่เว่น"]): return "Seven Eleven"
    if any(k in s for k in ["CJ", "ซี เจ", "ซีเจ", "Cj Express", "Cj MORE"]): return "CJ EXPRESS"
    if any(k in s for k in ["ท็อปส์ เดลี่", "ท็อปส์ มาร์เก็ต", "ท็อปส์มาร์เก็ต", "Tops", "ห้างท็อปส์", "TOPS"]): return "Tops"
    if any(k in s for k in ["Maxvalue", "แม็กซ์แวลู", "Max Valu"]): return "Maxvalue"
    # if "เดอะมอลล์" in s: return "The Mall"
    # if "แม็คโค" in s: return "Makro"
    if any(k in s for k in ["แม็คโค", "เเม็คโค"]): return "Makro"
    if any(k in s for k in ["กูรเมต์", "กูร์เมต์","กูร์เมต์"]): return "Gourmet Market"
    if any(k in s for k in ["เดอะมอลล์"]): return "The Mall"
    if any(k in s for k in ["เซนทรัล", "CENTRAL", "เซ็นทรัล"]): return "Central"
    if any(k in s for k in ["วัตสัน", "Watson"]): return "Watson"
    if any(k in s for k in ["เฟรชมาร์ท", "Freshmat"]): return "CP Freshmart"
    if "โฮมโปร" in s: return "HomePro"
    if "ร้านเอเซ่เว่น" in s: return "ร้านอื่นๆ"
    return "ร้านอื่นๆ"

df['ประเภทร้านค้า'] = df['SHOP_NAME'].apply(classify_shop_type)

# แก้ไขร้านค้ากรณีพิเศษ
mask_other = df['SHOP_NAME'].str.contains("ร้านหน้าโลตัส|Watson|วัตสัน|ปลาผา|WASH|Amazon|เอเมซอน|คาเฟ่|Cafe|ศูนย์อาหาร|อาหาร|ร้านยา|ขายยา|ข้าว|DRUG", case=False, na=False)
df.loc[mask_other, 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
df.loc[df['SHOP_NAME'].str.contains("โฮมโปร", case=False, na=False), 'ประเภทร้านค้า'] = "HomePro"

# แก้ไขตามรหัส
df.loc[df['CODE_7_DIGITS'].astype(str).str.startswith("4111", na=False), 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
df.loc[df['CODE_7_DIGITS'].astype(str).isin(["1250001", "1250002", "1250008", "1160015"]), 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
df.loc[df['SHOP_NAME'] == 'ท็อปน์ (เซ็นทรัล นนทบุรี)', 'ประเภทร้านค้า'] = "Tops"
df.loc[df['SHOP_NAME'] == 'ร้านเอเซ่เว่น', 'ประเภทร้านค้า'] = "ร้านอื่นๆ"

# ..ห้างบิ๊กซีมินิ..... บิ๊กซีมาร์เก็ต และโลตัสเอ็กเพรส ไม่สามารถใช้ราคากลางได้
# 1. Big C Mini
mask_bigc_mini = (df['ประเภทร้านค้า'] == 'Big C') & (df['SHOP_NAME'].str.contains('มินิ|Mini', case=False, na=False))
df.loc[mask_bigc_mini, 'ประเภทร้านค้า'] = 'BigC_mini'
# # 2. Big C Market
# mask_bigc_mkt = (df['ประเภทร้านค้า'] == 'Big C') & (df['SHOP_NAME'].str.contains('มาร์เก็ต|มาเก็ต|เกต|มาเกต|Market', case=False, na=False))
# df.loc[mask_bigc_mkt, 'ประเภทร้านค้า'] = 'BigC_market'
# 3. Lotus Express
mask_lotus_exp = (df['ประเภทร้านค้า'] == 'Tesco Lotus') & (df['SHOP_NAME'].str.contains('เฟรซ|เฟรช|โก|exp', case=False, na=False))
df.loc[mask_lotus_exp, 'ประเภทร้านค้า'] = 'Lotus_gofresh'
mask_lotus_exp = (df['ประเภทร้านค้า'] == 'Lotus_gofresh') & (df['SHOP_NAME'].str.contains('โก้', case=False, na=False))
df.loc[mask_lotus_exp, 'ประเภทร้านค้า'] = 'Tesco Lotus'

mask_Tops_day = (df['ประเภทร้านค้า'] == 'Tops') & (df['SHOP_NAME'].str.contains('เดลี่', case=False, na=False))
df.loc[mask_Tops_day, 'ประเภทร้านค้า'] = 'Tops_daily'


# ==========================================
# 3. จัดการภาค (Region)
# ==========================================
region_map = {
    'กลาง': ["CU", "CC"], 'เหนือ': ["NU", "NN"], 
    'ใต้': ["SU", "SS"], 'ตะวันออกเฉียงเหนือ': ["EU", "EE"]
}
df['ภาค'] = None
for region, codes in region_map.items():
    df.loc[df['TYPE'].isin(codes), 'ภาค'] = region

target_provinces = ['นนทบุรี', 'สมุทรปราการ', 'ปทุมธานี']
df.loc[df['PROVINCE_NAME'].isin(target_provinces), 'ภาค'] = 'ปริมณฑล'
df['ภาค'] = df['ภาค'].fillna('กทม.')
df['PROVINCE_NAME'] = df['PROVINCE_NAME'].replace('กรุงเทพมหานคร', 'กทม.')

# Dealer Group
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
df = df.merge(ds, on=["DILLER_CODE"], how="left")

# สร้าง Column GROUP
df['GROUP'] = df['PROVINCE_NAME'].fillna('').astype(str) + df['กลุ่ม'].fillna('').astype(str)


In [9]:
dx = df.copy()
dx['YM'] = dx['YEAR'].str[2:4]+ dx['MONTH']
# dx['COMM1'] = dx['COMM1'].astype(str) 
# dx['COMM1'] = dx['COMM1'].replace('nan', '')
# dx['COMM1'] = dx['COMM1'].fillna('')
# dx["COMM1"] = dx["COMM1"].astype(str).str.replace('=', '', regex=False)

# dx['COMM2'] = dx['COMM2'].astype(str) 
# dx['COMM2'] = dx['COMM2'].replace('nan', '')
# dx['COMM2'] = dx['COMM2'].fillna('')
# dx["COMM2"] = dx["COMM2"].astype(str).str.replace('nan', '', regex=False)

# dx = dx.fillna('')
# dx  = dx.replace(np.nan, '', regex=True)

pivot_df = dx.pivot_table(
    index=['TYPE', 'PROVINCE_NAME', 'GROUP','DESCRIPTION','SHOP_NAME','ประเภทร้านค้า', 'CODE_7_DIGITS','COMMODITY_CODE', 'DILLER_CODE'],
    columns=['YM'],
    values=['AVG', 'REL','COMPARE_PRICE'],    
    aggfunc={'AVG': 'first', 'REL': 'first','COMPARE_PRICE':'first'} # aggfunc={'COMM1': 'first'} # aggfunc='mean'
).reset_index()

pivot_df = pivot_df[['TYPE', 'PROVINCE_NAME', 'GROUP','DESCRIPTION','SHOP_NAME','ประเภทร้านค้า', 'CODE_7_DIGITS','COMMODITY_CODE', 'DILLER_CODE','AVG', 'REL','COMPARE_PRICE']]

month = excel_files[-1].split('_')[2].replace('.xlsx', '')  
prefix = excel_files[0].split('_')[0]  # 
filename = f"CPI_{month}_{prefix}_.xlsx"

In [10]:
# print(pivot_df.columns)
import re, numpy as np, pandas as pd

pivot_df.columns.names
# display(pivot_df.head(2))
# pivot_df['YM']
pivot_df.columns.get_level_values(0)
# pivot_df
# PD = pivot_df[['TYPE', 'SHOP_NAME', 'PROVINCE_NAME', 'DESCRIPTION', 'COMMODITY_CODE',
#        'DILLER_CODE','AVG','REL']].copy()

PD = pivot_df[['TYPE', 'GROUP','DESCRIPTION','SHOP_NAME','ประเภทร้านค้า', 'CODE_7_DIGITS','COMMODITY_CODE', 'DILLER_CODE','AVG', 'REL']].copy()
a1 = PD.columns.get_level_values(0)
a2 = PD.columns.get_level_values(1)

PD.columns = a1 + a2
# cols = PD.columns[27:]
# PD['Series_Max'] = PD[cols].max(axis=1)
# PD['Series_Min'] = PD[cols].min(axis=1)

# pivot_df['REL'].max(axis=1)


# เลือกคอลัมน์เดือนจาก PD (ไม่ใช่ df)
rel_cols = sorted(
    [c for c in PD.columns if re.fullmatch(r'REL\d{4}', str(c))],
    key=lambda x: int(str(x)[3:])
)

# เผื่อบางคอลัมน์เป็นสตริง -> แปลงเป็นตัวเลข (NaN ถ้าแปลงไม่ได้)
PD[rel_cols] = PD[rel_cols].apply(pd.to_numeric, errors='coerce')
PD['Rel_Max'] = PD[rel_cols].max(axis=1)
PD['Rel_Min'] = PD[rel_cols].min(axis=1)

def longest_streak(bool_arr: np.ndarray) -> int:
    if bool_arr.size == 0:
        return 0
    b = np.concatenate(([False], bool_arr, [False]))
    edges = np.flatnonzero(b[1:] != b[:-1])
    # ความยาวช่วง True ต่อเนื่อง
    lens = edges[1::2] - edges[0::2]
    return int(lens.max()) if lens.size else 0


def trailing_true_run(b: np.ndarray) -> int:
    """
    นับจำนวน True ต่อเนื่องจาก 'ท้ายสุด' ของอาเรย์ b (บูลีน)
    """
    if b.size == 0:
        return 0
    rev = b[::-1]
    idx = np.flatnonzero(~rev)  # ตำแหน่งแรกที่เป็น False เมื่อมองจากท้าย
    return len(b) if idx.size == 0 else int(idx[0])


# 1) สตรีคยาวสุดของ "มีข้อมูล (ไม่เป็น NaN)"
PD['Relstreak_max_nonnull'] = PD[rel_cols].notna().apply(lambda s: longest_streak(s.to_numpy()), axis=1)

# 2) สตรีคยาวสุดของ "ค่า == 100" (ถ้าต้องการ)
PD['Relstreak_max_eq100'] = PD[rel_cols].apply(lambda r: longest_streak((r.to_numpy() == 100)), axis=1)

# PD['stable3month'] = PD[rel_cols[-3:]].apply(lambda r: longest_streak((r.to_numpy() == 100)), axis=1)
# PD.loc[PD['stable3month'] != 3, 'stable3month'] = ''

# PD['stable6month'] = PD[rel_cols[-6:]].apply(lambda r: longest_streak((r.to_numpy() == 100)), axis=1)
# PD.loc[PD['stable6month'] != 6, 'stable6month'] = ''

# PD['stable9month'] = PD[rel_cols[-9:]].apply(lambda r: longest_streak((r.to_numpy() == 100)), axis=1)
# PD.loc[PD['stable9month'] != 9, 'stable9month'] = ''

# PD['stable12month'] = PD[rel_cols[-12:]].apply(lambda r: longest_streak((r.to_numpy() == 100)), axis=1)
# PD.loc[PD['stable12month'] != 12, 'stable12month'] = ''

# --- สตรีคถอยหลังสำหรับ REL ---

# 1) ถอยหลัง: มีข้อมูล (ไม่เป็น NaN) ต่อเนื่องนานเท่าไร
PD['Reltrail_nonnull'] = PD[rel_cols].apply(
    lambda r: trailing_true_run(~np.isnan(r.to_numpy(dtype=float))),
    axis=1
)

# 2) ถอยหลัง: ค่า == 100 ต่อเนื่องนานเท่าไร
PD['Reltrail_eq100'] = PD[rel_cols].apply(
    lambda r: trailing_true_run((pd.to_numeric(r, errors='coerce').to_numpy() == 100)),
    axis=1
)


def trailing_true_run(b: np.ndarray) -> int:
    """
    นับจำนวน True ต่อเนื่องจาก 'ท้ายสุด' ของอาเรย์ b (บูลีน)
    """
    if b.size == 0:
        return 0
    rev = b[::-1]
    idx = np.flatnonzero(~rev)  # ตำแหน่งแรกที่เป็น False เมื่อมองจากท้าย
    return len(b) if idx.size == 0 else int(idx[0])


# (ตัวเลือก) สตรีคยาวสุดของ "ค่า != 100"
# PD['streak_max_ne100'] = PD[rel_cols].apply(lambda r: longest_streak((r.to_numpy() != 100)), axis=1)
# เลือกและเรียงคอลัมน์ AVG ตามเวลา
avg_cols = sorted([c for c in PD.columns if re.fullmatch(r'AVG\d{4}', str(c))],
                  key=lambda x: int(x[3:]))

# ให้เป็นตัวเลขก่อน
PD[avg_cols] = PD[avg_cols].apply(pd.to_numeric, errors='coerce')

# หา diff เดือนต่อเดือนตามลำดับคอลัมน์
diffs = PD[avg_cols].diff(axis=1)

# ค่าเพิ่มขึ้นมากสุด และค่าลดลงมากสุด (ลบ)
PD['AVG_maxdiff'] = diffs.max(axis=1, skipna=True)
PD['AVG_mindiff'] = diffs.min(axis=1, skipna=True)

avg_cols = sorted(
    [c for c in PD.columns if re.fullmatch(r'AVG\d{4}', str(c))],
    key=lambda x: int(str(x)[3:])
)
PD['CountAVG'] = PD[avg_cols].count(axis=1)
PD['AVG_Max'] = PD[avg_cols].max(axis=1)
PD['AVG_Min'] = PD[avg_cols].min(axis=1)

import numpy as np
import pandas as pd

def tail_stable_run(row, tol=None, skip_trailing_nan=True):
    """
    นับความยาวรันของค่า 'คงเดิม' จากท้ายแถว (ค่าล่าสุด) ย้อนกลับไป
    - row: Series/array ของค่าตามเวลา (คอลัมน์เรียงจากเก่า -> ใหม่)
    - tol: ค่ากำหนดความใกล้เคียง (atol) ถ้าต้องการถือว่า 'เท่ากันแบบมี tolerance'
    - skip_trailing_nan: ถ้า True จะข้าม NaN ที่อยู่ท้ายสุดก่อนเริ่มนับ

    คืนค่าเป็นจำนวนคอลัมน์ที่ 'เท่ากับค่าล่าสุด' ติดต่อกัน
    """
    arr = pd.to_numeric(row, errors='coerce').to_numpy(dtype=float)
    i = len(arr) - 1

    # ข้าม NaN ท้ายแถว (เช่น ยังไม่ได้กรอกค่าล่าสุด)
    if skip_trailing_nan:
        while i >= 0 and np.isnan(arr[i]):
            i -= 1

    if i < 0:         # ทั้งแถวเป็น NaN
        return 0

    last = arr[i]     # ค่าล่าสุด (ตัว anchor)
    if np.isnan(last):
        return 0

    cnt = 1           # นับค่าล่าสุดตัวเองก่อน
    i -= 1
    while i >= 0 and not np.isnan(arr[i]):
        same = (np.isclose(arr[i], last, atol=tol) if tol is not None else arr[i] == last)
        if not same:
            break
        cnt += 1
        i -= 1

    return cnt

# ใช้งานกับตาราง PD
# 1. แปลงคอลัมน์ให้รองรับข้อความ (Text) ก่อน

PD['AVGtail_stable'] = PD[avg_cols].apply(lambda r: tail_stable_run(r), axis=1)
# PD.loc[PD['tail_stable'] , 'stable12month'] = ''

# mask = (PD['COMM1'].fillna('').str.strip().eq(''))
mask = PD[avg_cols[-1]].fillna('').str.strip().eq('')
# mask = PD[avg_cols[-1]].isna()
PD.loc[mask, 'AVGtail_stable'] = None   # ตั้งเป็นสตริงว่าง

# PD.loc[PD['stable12month'] != 12, 'stable12month'] = ''
# (ถ้าต้องการ “ความต่างมากสุดแบบไม่สนเครื่องหมาย”)
# PD['AVG_absmaxdiff'] = diffs.abs().max(axis=1, skipna=True)

insert_idx = PD.columns.get_loc('DILLER_CODE')+1 # 1. หาตำแหน่ง Index ของคอลัมน์ 'CODE_7_DIGITS'
val_to_insert = PD['COMMODITY_CODE'].astype(str) + PD['DILLER_CODE'].astype(str) # 2. แปลงเป็น String ก่อนเชื่อม (เพื่อกันมันบวกกันเป็นตัวเลข)
if 'ComKeyBasic' in PD.columns: PD.drop(columns=['ComKeyBasic'], inplace=True)
PD.insert(insert_idx, 'ComKeyBasic', val_to_insert) # 3. Insert ลงไปที่ตำแหน่งนั้น

# ... (โค้ดส่วนบนของคุณเหมือนเดิม) ...

 # <--- อย่าลืม import re ไว้บรรทัดบนสุดของไฟล์นะครับ

# # ... (โค้ดส่วนเตรียม DataFrame ของคุณ) ...
# import re
# import pandas as pd

output_file = "TimeSeries_FULL_GMonth.xlsx"

# ใช้ ExcelWriter
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    # 1. เขียนข้อมูล
    PD.to_excel(writer, sheet_name='Sheet1', index=False)
    
    workbook = writer.book
    worksheet = writer.sheets['Sheet1']
    
    # --- เตรียม Format ---
    header_fmt = workbook.add_format({
        'font_size': 8, 'bold': True, 'border': 1, 
        'text_wrap': False, 'valign': 'vcenter', 'align': 'left'
    })
    
    gray_fmt   = workbook.add_format({'bg_color': '#D3D3D3'})
    blue_fmt   = workbook.add_format({'bg_color': '#E0F7FA'})
    yellow_fmt = workbook.add_format({'bg_color': '#FFFFE0'})

    # ------------------------------------------------------------------
    # [จุดสำคัญ] รวมการตั้งค่า (กว้าง, สี, ซ่อน) ไว้ใน Dict เดียว
    # ------------------------------------------------------------------
    col_config = {
        # คอลัมน์ที่ต้องการใส่สีเทา + กำหนดความกว้าง
        'TYPE':            {'width': 2,  'fmt': gray_fmt},
        'GROUP':           {'width': 5,  'fmt': gray_fmt},
        'DESCRIPTION':     {'width': 20, 'fmt': gray_fmt},
        'SHOP_NAME':       {'width': 10, 'fmt': gray_fmt},
        'ประเภทร้านค้า':      {'width': 10, 'fmt': gray_fmt},
        'ComKeyBasic':     {'width': 26, 'fmt': gray_fmt},
        
        # คอลัมน์ที่ต้องการซ่อน (Hide)
        'CODE_7_DIGITS':   {'hidden': True},
        'COMMODITY_CODE':  {'hidden': True},
        'DILLER_CODE':     {'hidden': True}
    }

    # --- วนลูปเดียวจบ ครบทุกเงื่อนไข ---
    for i, col in enumerate(PD.columns):
        # 1. เขียน Header ทับ (ปรับขนาด Font/Align)
        worksheet.write(0, i, col, header_fmt)

        # 2. เช็คว่าเป็นคอลัมน์พิเศษที่ระบุใน Dict หรือไม่
        if col in col_config:
            cfg = col_config[col]
            
            # ดึงค่า Config (ถ้าไม่มีให้เป็น None)
            w = cfg.get('width', None)
            f = cfg.get('fmt', None)
            
            # เช็คว่าต้องซ่อนหรือไม่
            options = {'hidden': True} if cfg.get('hidden') else {}
            
            # สั่ง set_column ครั้งเดียว
            worksheet.set_column(i, i, w, f, options)

        # 3. เช็คเงื่อนไข Regex สำหรับ AVG (สีฟ้า)
        elif re.match(r'^AVG\d{4}$', str(col)):
            worksheet.set_column(i, i, 10, blue_fmt)
            
        # 4. เช็คเงื่อนไข Regex สำหรับ REL (สีเหลือง)
        elif re.match(r'^REL\d{4}$', str(col)):
            worksheet.set_column(i, i, 10, yellow_fmt)

    # --- ตั้งค่า View ---
    worksheet.freeze_panes(1, 9) 
    worksheet.autofilter(0, 0, PD.shape[0], PD.shape[1] - 1)

print(f"บันทึกไฟล์ {output_file} เรียบร้อยแล้ว (ซ่อนคอลัมน์เรียบร้อย)")

บันทึกไฟล์ TimeSeries_FULL_GMonth.xlsx เรียบร้อยแล้ว (ซ่อนคอลัมน์เรียบร้อย)


In [11]:
PD.columns

Index(['TYPE', 'GROUP', 'DESCRIPTION', 'SHOP_NAME', 'ประเภทร้านค้า',
       'CODE_7_DIGITS', 'COMMODITY_CODE', 'DILLER_CODE', 'ComKeyBasic',
       'AVG6601', 'AVG6602', 'AVG6603', 'AVG6604', 'AVG6605', 'AVG6606',
       'AVG6607', 'AVG6608', 'AVG6609', 'AVG6610', 'AVG6611', 'AVG6612',
       'AVG6701', 'AVG6702', 'AVG6703', 'AVG6704', 'AVG6705', 'AVG6706',
       'AVG6707', 'AVG6708', 'AVG6709', 'AVG6710', 'AVG6711', 'AVG6712',
       'AVG6801', 'AVG6802', 'AVG6803', 'AVG6804', 'AVG6805', 'AVG6806',
       'AVG6807', 'AVG6808', 'AVG6809', 'AVG6810', 'AVG6811', 'AVG6812',
       'AVG6901', 'AVG6902', 'REL6601', 'REL6602', 'REL6603', 'REL6604',
       'REL6605', 'REL6606', 'REL6607', 'REL6608', 'REL6609', 'REL6610',
       'REL6611', 'REL6612', 'REL6701', 'REL6702', 'REL6703', 'REL6704',
       'REL6705', 'REL6706', 'REL6707', 'REL6708', 'REL6709', 'REL6710',
       'REL6711', 'REL6712', 'REL6801', 'REL6802', 'REL6803', 'REL6804',
       'REL6805', 'REL6806', 'REL6807', 'REL6808', 'REL6

In [12]:
'SeriesU'+filename[4:10]+'.xlsx'
# 1. หาตำแหน่งเริ่มต้นของคำว่า "256"
start_index = filename.find("256")

# 2. หาตำแหน่งจุด (.) สุดท้าย (ใช้ rfind เผื่อชื่อไฟล์มีจุดอื่นคั่น)
end_index = filename.rfind("c")

# 3. ตรวจสอบว่าเจอทั้งคู่ไหม แล้วตัดคำ
if start_index != -1 and end_index != -1:
    # ตัดตั้งแต่ start_index ถึง end_index
    core_name = filename[start_index:end_index]
    
    # เอามาประกอบร่าง
    # final_name = 'SeriesG' + core_name + '.xlsx'
    final_name = 'SeriesGMonth' + '.xlsx'
    
    print(final_name)
else:
    print("ไม่พบรูปแบบ '256' หรือ '.' ในชื่อไฟล์")

PD['TimeCheck'] = filename[start_index:end_index]

PD[['TYPE', 'GROUP','DESCRIPTION','SHOP_NAME','ประเภทร้านค้า', 'CODE_7_DIGITS','COMMODITY_CODE', 'DILLER_CODE'
    ,'CountAVG','AVG_Max','AVG_Min','AVGtail_stable', 'AVG_maxdiff','AVG_mindiff','Rel_Max','Rel_Min'
    ,'Relstreak_max_nonnull','Relstreak_max_eq100','Reltrail_nonnull'
    ,'Reltrail_eq100','TimeCheck']].to_excel(final_name,index=False)  

SeriesGMonth.xlsx


In [13]:
# --- จบการจับเวลา ---
end_time = time.time()

# คำนวณเวลาที่ใช้
execution_time = end_time - start_time
print(f"Total execution time: {execution_time:.2f} seconds")

Total execution time: 264.61 seconds


In [14]:
# import time
# import pandas as pd
# import os
# import gc
# from pathlib import Path

# def run_optimized_pipeline():
#     start_time = time.time()
#     current_dir = Path.cwd() 
#     # ระบุ Path ไปยังโฟลเดอร์ Data_G_Month
#     folder_path = current_dir.parent / "01-database" / "Data_G_Month"
    
#     # --- 1. การจัดการไฟล์ (Filtering & Sorting) ---
#     unwanted = {'pivot_compare_PriceRelComm.xlsx', 'Test.xlsx', 'Test3.xlsx', 'SeriesG102568.xslx', 'TimeSeries_FULL.xlsx'}
#     excel_files = [
#         f for f in os.listdir(folder_path) 
#         if f.lower().startswith('cpig') and f.endswith(".xlsx") and f not in unwanted
#     ]
    
#     # เรียงลำดับไฟล์ตาม ปี-เดือน ที่อยู่ท้ายชื่อไฟล์
#     excel_files = sorted(excel_files, key=lambda x: tuple(map(int, x.replace('.xlsx', '').split('_')[-1].split('-'))))
#     excel_files = excel_files[:-1] # ตัดไฟล์ล่าสุดออกตาม logic เดิมของคุณ

#     # --- 2. ตั้งค่าการอ่านไฟล์แบบประหยัด RAM ---
#     selected_columns = [
#         'TYPE', 'G_FLAG', 'L_FLAG', 'R_FLAG', 'SHOP_NAME', 'PROVINCE_NAME',
#         'DESCRIPTION', 'COMMODITY_CODE', 'DILLER_CODE', 'MONTH', 'YEAR', 'AVG', 'REL','COMPARE_PRICE'
#     ]

#     # ใช้ 'category' เพื่อลด RAM และบังคับรหัสต่างๆ เป็น String
#     col_types = {
#         'TYPE': 'category',
#         'PROVINCE_NAME': 'category',
#         'MONTH': str, 
#         'YEAR': str, 
#         'COMMODITY_CODE': str, 
#         'DILLER_CODE': str
#     }

#     dataframes = []
#     print(f"🚀 Processing {len(excel_files)} files with Calamine Engine...")

#     # --- 3. Loop อ่านไฟล์ด้วย Calamine ---
#     for file in excel_files:
#         file_path = folder_path / file
#         try:
#             # ใช้ engine='calamine' เพื่อความเร็วสูงสุดในการอ่าน
#             df = pd.read_excel(
#                 file_path, 
#                 usecols=selected_columns, 
#                 dtype=col_types, 
#                 engine='calamine'
#             )
            
#             # ลบแถวที่ว่างเปล่าทันทีเพื่อประหยัด RAM ก่อนนำไป concat
#             df.dropna(subset=['AVG'], inplace=True)
#             dataframes.append(df)
#             print(f"   ✅ Loaded: {file}")
#         except Exception as e:
#             print(f"   ⚠️ Error loading {file}: {e}")

#     # --- 4. การรวมข้อมูลและเคลียร์ RAM ---
#     if dataframes:
#         combined_df = pd.concat(dataframes, ignore_index=True)
        
#         # ลบ List ทิ้งและสั่ง Garbage Collect ทันทีเพื่อคืน RAM
#         del dataframes
#         gc.collect()
        
#         # จัดการ Format ข้อมูลแบบ Vectorized (รวดเร็ว)
#         combined_df['MONTH'] = combined_df['MONTH'].str.zfill(2)
#         combined_df['COMMODITY_CODE'] = combined_df['COMMODITY_CODE'].str.zfill(16)
#         combined_df['DILLER_CODE'] = combined_df['DILLER_CODE'].str.zfill(10)

#         # --- 5. การเรียงลำดับแบบ Inplace (ไม่สร้าง Copy ใหม่) ---
#         print(f"📊 Sorting {len(combined_df):,} rows...")
#         combined_df.sort_values(
#             by=['TYPE', 'COMMODITY_CODE', 'DILLER_CODE', 'YEAR', 'MONTH'],
#             inplace=True
#         )
#         combined_df.reset_index(drop=True, inplace=True)

#         print("-" * 30)
#         print(f"✨ SUCCESS! Combined Rows: {len(combined_df):,}")
#         print(f"⏱️ total time: {time.time() - start_time:.2f} seconds")
        
#         return combined_df
#     else:
#         print("❌ No data found.")
#         return None

# # เรียกใช้งาน
# combined_df = run_optimized_pipeline()